<a href="https://colab.research.google.com/github/feliciadominic14-create/Homework/blob/main/Felicia_StudentExam_Encoding_Imbalance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import numpy as np
import pandas as pd

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

In [35]:
df = pd.read_csv("/content/student_exam_performance.csv")

In [36]:
print("Initial shape:", df.shape)
print("\nColumns and dtypes:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())
print("\nTarget distribution:\n", df["pass_status"].value_counts(dropna=False))

Initial shape: (100000, 44)

Columns and dtypes:
 student_id                     object
age                             int64
gender                         object
education_level                object
school_type                    object
family_income                  object
parent_education               object
urban_rural                    object
previous_exam_score           float64
previous_gpa                  float64
attendance_percentage         float64
assignment_completion_rate    float64
class_participation            object
study_hours_per_day           float64
self_study_hours              float64
private_tuition                 int64
online_learning_hours         float64
study_consistency              object
study_environment              object
study_method                   object
revision_frequency             object
practice_tests_completed        int64
notes_quality                  object
sleep_hours                   float64
sleep_quality                  object


In [37]:
potential_leakage_cols = [
    col for col in df.columns
    if any(k in col.lower() for k in ["score", "grade", "mark", "total", "percent", "id"])
    and col != "pass_status"
]
print("\nPotential leakage/ID columns to review:", potential_leakage_cols)


Potential leakage/ID columns to review: ['student_id', 'previous_exam_score', 'attendance_percentage', 'time_management_score', 'exam_score', 'performance_grade']


In [38]:
cols_to_drop = [c for c in potential_leakage_cols if c in df.columns]
print("Dropping columns:", cols_to_drop)

df_clean = df.drop(columns=cols_to_drop).copy()

Dropping columns: ['student_id', 'previous_exam_score', 'attendance_percentage', 'time_management_score', 'exam_score', 'performance_grade']


In [39]:
target_col = "pass_status"

In [40]:
X = df_clean.drop(columns=[target_col])
y_raw = df_clean[target_col]

In [41]:
mask = pd.notna(y_raw)
X_clean = X[mask] if hasattr(X, "iloc") else X[np.asarray(mask)]
y_clean = y_raw[mask] if hasattr(y_raw, "iloc") else np.asarray(y_raw)[np.asarray(mask)]

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

In [42]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (80000, 37)
X_test shape: (20000, 37)


In [43]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

In [44]:
num_imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

In [45]:
X_train_num_imputed = num_imputer.fit_transform(X_train[num_cols])
X_test_num_imputed = num_imputer.transform(X_test[num_cols])

In [46]:
X_train_num_scaled = scaler.fit_transform(X_train_num_imputed)
X_test_num_scaled = scaler.transform(X_test_num_imputed)

In [47]:
num_train_df = pd.DataFrame(X_train_num_scaled, columns=num_cols, index=X_train.index)
num_test_df = pd.DataFrame(X_test_num_scaled, columns=num_cols, index=X_test.index)

In [48]:
cat_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
print("Categorical columns found:", cat_cols)

Categorical columns found: ['gender', 'education_level', 'school_type', 'family_income', 'parent_education', 'urban_rural', 'class_participation', 'study_consistency', 'study_environment', 'study_method', 'revision_frequency', 'notes_quality', 'sleep_quality', 'break_frequency', 'motivation_level', 'device_availability', 'educational_app_usage', 'exam_difficulty', 'performance_level']


In [49]:
ordinal_mapping = {}
for col in cat_cols:
    vals = sorted(X_train[col].dropna().unique().tolist())
    if any(k in col.lower() for k in ["education", "level", "frequency", "income", "status"]):
        ordinal_mapping[col] = vals

In [50]:
ordinal_cols = list(ordinal_mapping.keys())
onehot_cols = [c for c in cat_cols if c not in ordinal_cols]

In [51]:
cat_imputer = SimpleImputer(strategy="most_frequent")
X_train_cat_imp = pd.DataFrame(
    cat_imputer.fit_transform(X_train[cat_cols]),
    columns=cat_cols,
    index=X_train.index,
)
X_test_cat_imp = pd.DataFrame(
    cat_imputer.transform(X_test[cat_cols]),
    columns=cat_cols,
    index=X_test.index,
)

In [52]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
if len(onehot_cols) > 0:
    ohe_train_arr = ohe.fit_transform(X_train_cat_imp[onehot_cols])
    ohe_test_arr = ohe.transform(X_test_cat_imp[onehot_cols])
    ohe_feature_names = ohe.get_feature_names_out(onehot_cols)
    ohe_train_df = pd.DataFrame(ohe_train_arr, columns=ohe_feature_names, index=X_train.index)
    ohe_test_df = pd.DataFrame(ohe_test_arr, columns=ohe_feature_names, index=X_test.index)
else:
    ohe_train_df = pd.DataFrame(index=X_train.index)
    ohe_test_df = pd.DataFrame(index=X_test.index)

In [53]:
if len(ordinal_cols) > 0:
    ordered_lists = [ordinal_mapping[c] for c in ordinal_cols]
    ord_enc = OrdinalEncoder(
        categories=ordered_lists,
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )
    ord_train_arr = ord_enc.fit_transform(X_train_cat_imp[ordinal_cols])
    ord_test_arr = ord_enc.transform(X_test_cat_imp[ordinal_cols])
    ord_train_df = pd.DataFrame(ord_train_arr, columns=ordinal_cols, index=X_train.index)
    ord_test_df = pd.DataFrame(ord_test_arr, columns=ordinal_cols, index=X_test.index)
else:
    ord_train_df = pd.DataFrame(index=X_train.index)
    ord_test_df = pd.DataFrame(index=X_test.index)

In [54]:
cat_encoded_train_df = pd.concat([ohe_train_df, ord_train_df], axis=1)
cat_encoded_test_df = pd.concat([ohe_test_df, ord_test_df], axis=1)

In [55]:
X_train_prepared = pd.concat([num_train_df, cat_encoded_train_df], axis=1)
X_test_prepared = pd.concat([num_test_df, cat_encoded_test_df], axis=1)

In [56]:
print("X_train_prepared shape:", X_train_prepared.shape)
print("X_test_prepared shape:", X_test_prepared.shape)

X_train_prepared shape: (80000, 61)
X_test_prepared shape: (20000, 61)


In [57]:
target_map = {"Fail": 0, "Pass": 1}

In [58]:
y_train = y_train_raw.map(target_map)
y_test = y_test_raw.map(target_map)

In [59]:
if y_train.isna().any() or y_test.isna().any():
    normalized_map = {k.lower(): v for k, v in target_map.items()}
    y_train = y_train_raw.astype(str).str.strip().str.lower().map(normalized_map)
    y_test = y_test_raw.astype(str).str.strip().str.lower().map(normalized_map)

In [60]:
print("y_train value counts:\n", y_train.value_counts())
print("y_test value counts:\n", y_test.value_counts())

y_train value counts:
 pass_status
1    61911
0    18089
Name: count, dtype: int64
y_test value counts:
 pass_status
1    15478
0     4522
Name: count, dtype: int64


In [64]:
rng = np.random.default_rng(42)
counts = y_train.value_counts()
max_count = counts.max()

In [61]:
print("Before resampling - y_train counts:\n", y_train.value_counts().to_dict())

Before resampling - y_train counts:
 {1: 61911, 0: 18089}


In [65]:
sampled_idx = np.concatenate([
    rng.choice(y_train[y_train == cls].index, size=max_count - cnt, replace=True)
    for cls, cnt in counts.items() if max_count > cnt
])

In [66]:
try:
    from imblearn.over_sampling import SMOTE

    smote = SMOTE(random_state=42)
    X_train_final, y_train_final = smote.fit_resample(X_train_prepared, y_train)
    X_train_final = pd.DataFrame(X_train_final, columns=X_train_prepared.columns)
    y_train_final = pd.Series(y_train_final, name=target_col)
    print("Applied SMOTE to training data.")
except ImportError:
    minority_class = y_train.value_counts().idxmin()
    majority_count = y_train.value_counts().max()
    minority_idx = y_train[y_train == minority_class].index
    sampled_idx = np.random.default_rng(42).choice(minority_idx,size=majority_count - len(minority_idx),replace=True)

Applied SMOTE to training data.


In [67]:
X_train_final = pd.concat(
    [X_train_prepared, X_train_prepared.loc[sampled_idx]], axis=0
).reset_index(drop=True)

y_train_final = pd.concat(
    [y_train, y_train.loc[sampled_idx]], axis=0
).reset_index(drop=True)

In [68]:
X_test_final = X_test_prepared.copy()
y_test_final = y_test.copy()

In [69]:
print("After resampling - y_train_final counts:\n", y_train_final.value_counts().to_dict())

After resampling - y_train_final counts:
 {1: 61911, 0: 61911}
